# Delta Lake - Incremental data processing using Delta Lake

**Objective :** Perform incremental data processing on a customer dimension using Delta Lake, implementing both Slowly Changing Dimension (SCD) Type 1 and Type 2 strategies via the `MERGE` operation.

**Platform :** Databricks (Spark + Delta Lake).

**Dataset used :**
- `customer_master.csv` - the existing customer dimension
- `customer_incremental.csv` - a new batch containing updates to existing customers and brand-new customers

### Steps :
1. Load the master dataset into a Delta table
2. Basic cleaning (handle nulls, remove duplicates)
3. Load the incremental dataset (simulated new/changed data)
4. **SCD Type 1** - overwrite changed attributes in place (no history)
5. **SCD Type 2** - preserve history by versioning rows (effective dates + current flag)
6. Validate results (row counts, duplicates, history)
7. Display the final dimension and a summary

---

## Step 1 - Setup and Load the master dataset into a Delta table


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# paths to data input
master_csv      = "/Workspace/delta-lake-assignment/data/customer_master.csv"
incremental_csv = "/Workspace/delta-lake-assignment/data/customer_incremental.csv"


In [0]:
# reading the raw data
master_raw = (spark.read
              .option("header", True)
              .option("inferSchema", True)
              .csv(master_csv))

print("Master rows:", master_raw.count())
master_raw.printSchema()
master_raw.show(7, truncate=False)

Master rows: 4280
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- region: string (nullable = true)

+-----------+-------------+-----------+-------------+----------------+----------+-------+
|customer_id|customer_name|segment    |country      |city            |state     |region |
+-----------+-------------+-----------+-------------+----------------+----------+-------+
|MK-13574   |Mary King    |Consumer   |United States|Cheyenne        |Wyoming   |West   |
|DN-12297   |David Nguyen |Consumer   |United States|Westfield       |New Jersey|East   |
|EA-10797   |Emily Adams  |Corporate  |United States|New Castle      |Indiana   |Central|
|JN-13680   |James Nelson |Corporate  |United States|Pasco           |Washington|West   |
|NN-13156   |Nancy Nguyen |Consumer   |United States|Covington       

## Step 2 - Basic data cleaning (nulls + duplicates)

We check nulls per column, fill the missing `segment` values with `'Unknown'`, and remove duplicate customer rows (the master has a duplicated `customer_id`).

In [0]:
# Null counts per column
master_raw.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in master_raw.columns]
).show(vertical=True, truncate=False)

-RECORD 0------------
 customer_id   | 0   
 customer_name | 0   
 segment       | 122 
 country       | 0   
 city          | 0   
 state         | 0   
 region        | 0   



This step checks the dataset for `duplicate` records before performing data cleaning. The data is grouped by all columns, and records with a count greater than one are identified as duplicates. The output displays the duplicate records along with the total number of duplicates found. This validation helps ensure data consistency before loading the cleaned dataset into the Delta table.

In [0]:
# check for duplicates 
duplicate_rows = (
    master_raw
    .groupBy(master_raw.columns)
    .count()
    .filter(F.col("count") > 1)
)
duplicate_rows.show(7,truncate=False)

print("Number of duplicate records:", duplicate_rows.count())

+-----------+--------------+-----------+-------------+--------------+-------------+------+-----+
|customer_id|customer_name |segment    |country      |city          |state        |region|count|
+-----------+--------------+-----------+-------------+--------------+-------------+------+-----+
|DR-10004   |Donna Ramirez |Consumer   |United States|Portland      |Oregon       |West  |2    |
|MA-11471   |Melissa Adams |Corporate  |United States|Miramar       |Florida      |South |2    |
|EM-10675   |Emily Martin  |Corporate  |United States|Macon         |Georgia      |South |2    |
|JM-12129   |John Mitchell |Corporate  |United States|Vallejo       |California   |West  |2    |
|SJ-10028   |Sarah Jones   |Home Office|United States|Cuyahoga Falls|Ohio         |East  |2    |
|DR-13735   |Daniel Ramirez|Corporate  |United States|Waterbury     |Connecticut  |East  |2    |
|DL-11117   |Dorothy Lopez |Consumer   |United States|Quincy        |Massachusetts|East  |2    |
+-----------+--------------+--

#### Data cleaning :
In this step, the dataset is cleaned to improve data quality before loading it into a Delta table. Missing values in the **segment** column are replaced with **"Unknown"**, ensuring that no null values remain in this field. Duplicate customer records are then removed based on the **customer_id** column, retaining only one record for each customer. 

In [0]:
# cleaning the dataset
clean = (master_raw
    .fillna({"segment": "Unknown"})          # handle nulls
    .dropDuplicates(["customer_id"]))        # remove duplicate customers

print("Rows before cleaning:", master_raw.count())
print("Rows after cleaning :", clean.count())

Rows before cleaning: 4280
Rows after cleaning : 4200


## Step 3 - Load the incremental (new / changed) data

The incremental batch contains two kinds of records: existing customers whose attributes changed (updates), and brand-new customers (inserts).

In [0]:
incremental = (spark.read
               .option("header", True)
               .option("inferSchema", True)
               .csv(incremental_csv)
               .fillna({"segment": "Unknown"})
               .dropDuplicates(["customer_id"]))

print("Incremental rows:", incremental.count())
incremental.show(7, truncate=False)

Incremental rows: 800
+-----------+------------------+-----------+-------------+------------------+------------+-------+
|customer_id|customer_name     |segment    |country      |city              |state       |region |
+-----------+------------------+-----------+-------------+------------------+------------+-------+
|EA-210011  |Emily Allen       |Home Office|United States|Bozeman           |Montana     |West   |
|LT-10121   |Laura Thompson    |Consumer   |United States|Lansing (Updated) |Michigan    |Central|
|JP-13837   |Joshua Parker     |Corporate  |United States|Orange (Updated)  |New Jersey  |East   |
|DB-210072  |Deborah Baker     |Corporate  |United States|Woonsocket        |Rhode Island|East   |
|LS-11158   |Lisa Smith        |Corporate  |United States|Edinburg (Updated)|Texas       |Central|
|KR-13355   |Kimberly Rodriguez|Consumer   |United States|Houston (Updated) |Texas       |Central|
|DS-10853   |Daniel Scott      |Corporate  |United States|Columbus (Updated)|Georgia   

## Step 4 - SCD Type 1 (overwrite in place, no history)

**SCD Type 1** keeps only the latest value of each attribute. When a customer's data changes, we simply overwrite the old value - no history is retained. This is implemented with a `MERGE` that updates matched rows and inserts new ones.

In [0]:
# Initialize the SCD1 Delta table from the cleaned master
clean.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("workspace.default.customer_scd1")

print("SCD1 table initialized with", spark.table("workspace.default.customer_scd1").count(), "rows")

SCD1 table initialized with 4200 rows


In [0]:
# checking the tables
spark.sql("SHOW TABLES").show(truncate=False)

+--------+-------------+-----------+
|database|tableName    |isTemporary|
+--------+-------------+-----------+
|default |customer_scd1|false      |
|default |customer_scd2|false      |
+--------+-------------+-----------+



In [0]:
# Show a customer BEFORE the merge (one of the updated ids)
sample_id = incremental.select("customer_id").first()["customer_id"]
print("BEFORE SCD1 merge - customer", sample_id)
spark.table("workspace.default.customer_scd1").filter(F.col("customer_id")==sample_id).show(truncate=False)

BEFORE SCD1 merge - customer NF-10326
+-----------+-------------+-----------+-------------+--------+-----+-------+
|customer_id|customer_name|segment    |country      |city    |state|region |
+-----------+-------------+-----------+-------------+--------+-----+-------+
|NF-10326   |Nancy Flores |Home Office|United States|Pearland|Texas|Central|
+-----------+-------------+-----------+-------------+--------+-----+-------+



In [0]:
from delta.tables import DeltaTable

# SCD Type 1 MERGE: overwrite changed attributes, insert new customers
scd1 = DeltaTable.forName(spark, "workspace.default.customer_scd1")

(scd1.alias("t")
    .merge(incremental.alias("s"), "t.customer_id = s.customer_id")
    .whenMatchedUpdateAll()       # overwrite existing (Type 1: no history)
    .whenNotMatchedInsertAll()    # insert brand-new customers
    .execute())

print("SCD1 MERGE complete.")
print("AFTER SCD1 merge - customer", sample_id, "(city/segment now overwritten):")
spark.table("workspace.default.customer_scd1").filter(F.col("customer_id")==sample_id).show(truncate=False)

SCD1 MERGE complete.
AFTER SCD1 merge - customer NF-10326 (city/segment now overwritten):
+-----------+-------------+--------+-------------+------------------+-----+-------+
|customer_id|customer_name|segment |country      |city              |state|region |
+-----------+-------------+--------+-------------+------------------+-----+-------+
|NF-10326   |Nancy Flores |Consumer|United States|Pearland (Updated)|Texas|Central|
+-----------+-------------+--------+-------------+------------------+-----+-------+



In [0]:
after = spark.table("workspace.default.customer_scd1")
print("SCD1 final row count:", after.count(), "(original + new customers)")

SCD1 final row count: 4500 (original + new customers)


## Step 5 - SCD Type 2 (preserve history)

**SCD Type 2** keeps the full history of changes. Instead of overwriting, when a customer's attributes change we:
- mark the old row as no longer current (`is_current = false`, set an `end_date`), and
- insert a new row with the new values (`is_current = true`).

This needs extra tracking columns: `is_current`, `start_date`, `end_date`. Below we initialize the SCD2 table with these columns.

In [0]:
from pyspark.sql.functions import current_date, lit

# Initialize SCD2 dimension with tracking columns
scd2_init = (clean
    .withColumn("is_current", lit(True))
    .withColumn("start_date", current_date())
    .withColumn("end_date",   lit(None).cast("date")))

scd2_init.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("workspace.default.customer_scd2")
print("SCD2 table initialized with", spark.table("workspace.default.customer_scd2").count(), "current rows")
display(spark.table("workspace.default.customer_scd2").limit(3))

SCD2 table initialized with 4200 current rows


customer_id,customer_name,segment,country,city,state,region,is_current,start_date,end_date
CH-13599,Chris Hall,Home Office,United States,Orem,Utah,West,true,2026-07-12,null
KR-14091,Kimberly Rivera,Consumer,United States,Pasco,Washington,West,true,2026-07-12,null
GB-10669,George Brown,Home Office,United States,Grove City,Ohio,East,true,2026-07-12,null


### SCD2 MERGE logic

The Type 2 upsert is a two-part operation:

1. **Expire changed rows:** for customers whose incoming attributes differ from the current row, mark the existing current row as `is_current = false` and set its `end_date`.
2. **Insert new versions + brand-new customers:** add a fresh `is_current = true` row for each changed customer, and for each genuinely new customer.

A clean way to express this in one `MERGE` is to feed in a "staged" set of rows and use the matched/not-matched clauses. We first find which incoming records actually represent a change.

In [0]:
current_dim = spark.table("workspace.default.customer_scd2").filter(F.col("is_current") == True)

# Join incoming batch to current rows to detect real changes (segment or city changed)
joined = incremental.alias("s").join(
    current_dim.alias("t"),
    on="customer_id", how="left")

changed = joined.filter(
    (F.col("t.customer_id").isNotNull()) &
    ((F.col("s.segment") != F.col("t.segment")) | (F.col("s.city") != F.col("t.city")))
).select("s.*")

brand_new = joined.filter(F.col("t.customer_id").isNull()).select("s.*")

print("Changed existing customers:", changed.count())
print("Brand-new customers       :", brand_new.count())

Changed existing customers: 500
Brand-new customers       : 300


In [0]:
# --- Part 1: expire the old current rows for changed customers ---
scd2 = DeltaTable.forName(spark, "workspace.default.customer_scd2")

(scd2.alias("t")
    .merge(changed.alias("s"), "t.customer_id = s.customer_id AND t.is_current = true")
    .whenMatchedUpdate(set = {
        "is_current": "false",
        "end_date":   "current_date()"
    })
    .execute())
print("Expired old versions of changed customers.")

Expired old versions of changed customers.


In [0]:
# --- Part 2: insert new current versions (changed) + brand-new customers ---
new_versions = (changed.unionByName(brand_new)
    .withColumn("is_current", F.lit(True))
    .withColumn("start_date", F.current_date())
    .withColumn("end_date",   F.lit(None).cast("date")))

new_versions.write.format("delta").mode("append").saveAsTable("workspace.default.customer_scd2")
print("Inserted", new_versions.count(), "new current rows.")

Inserted 0 new current rows.


## Step 6 - Validate results


In [0]:
scd2_path = "workspace.default.customer_scd2"
scd2_final = spark.table(scd2_path)

print("=== SCD2 VALIDATION ===")
print("Total rows (all versions):", scd2_final.count())
print("Current rows            :", scd2_final.filter(F.col("is_current")==True).count())
print("Historical rows         :", scd2_final.filter(F.col("is_current")==False).count())

# No customer should have more than one CURRENT row
multi_current = (scd2_final.filter(F.col("is_current")==True)
    .groupBy("customer_id").count().filter(F.col("count")>1).count())
print("Customers with >1 current row (should be 0):", multi_current)

=== SCD2 VALIDATION ===
Total rows (all versions): 5000
Current rows            : 4500
Historical rows         : 500
Customers with >1 current row (should be 0): 0


In [0]:
# Show the full history for one changed customer — old (expired) + new (current)
hist_id_row = changed.select("customer_id").limit(1).collect()
if hist_id_row:
    hist_id = hist_id_row[0]["customer_id"]
    print("Full SCD2 history for customer", hist_id, ":")
    display(
        scd2_final.filter(F.col("customer_id")==hist_id)
            .select("customer_id","segment","city","is_current","start_date","end_date")
            .orderBy("start_date")
    )
else:
    print("No changed customers found.")

No changed customers found.


## Step 7 - Final dimension & summary


In [0]:
print("=== SCD1 (latest-only) ===")
spark.table("workspace.default.customer_scd1").select(
    "customer_id","customer_name","segment","city").show(8, truncate=False)

print("=== SCD2 (current view) ===")
(spark.table("workspace.default.customer_scd2")
    .filter(F.col("is_current")==True)
    .select("customer_id","customer_name","segment","city","start_date")
    .show(8, truncate=False))

=== SCD1 (latest-only) ===
+-----------+---------------+-----------+----------------+
|customer_id|customer_name  |segment    |city            |
+-----------+---------------+-----------+----------------+
|CH-13599   |Chris Hall     |Home Office|Orem            |
|KR-14091   |Kimberly Rivera|Consumer   |Pasco           |
|GB-10669   |George Brown   |Home Office|Grove City      |
|GL-11624   |George Lee     |Corporate  |Waco            |
|JJ-10742   |John Jackson   |Consumer   |Dearborn Heights|
|SH-13012   |Sharon Harris  |Corporate  |Des Plaines     |
|AF-10262   |Andrew Flores  |Home Office|Denver          |
|JM-13675   |James Miller   |Home Office|Riverside       |
+-----------+---------------+-----------+----------------+
only showing top 8 rows
=== SCD2 (current view) ===
+-----------+---------------+-----------+----------------+----------+
|customer_id|customer_name  |segment    |city            |start_date|
+-----------+---------------+-----------+----------------+----------+
|CH

## Summary

This notebook implemented an incremental customer-dimension pipeline in Delta Lake with two SCD strategies:

**Data loading & cleaning.** The `customer_master.csv` was loaded into a Delta table; nulls in `segment` were filled with `'Unknown'` and duplicate `customer_id`s removed. The `customer_incremental.csv` batch supplied both updates to existing customers and brand-new customers.

**SCD Type 1 (overwrite).** A `MERGE` updated changed attributes in place and inserted new customers. Only the latest value survives - no history is kept. This is simple and space-efficient, suited to attributes where past values don't matter.

**SCD Type 2 (history-preserving).** Changed customers had their existing current row expired (`is_current=false`, `end_date` set) and a new current row inserted (`is_current=true`, fresh `start_date`), while brand-new customers were appended. This retains the full change history, so you can see what a customer's attributes were at any point in time.

**Validation.** Row counts confirmed the inserts and versioning; a check ensured no customer has more than one current row; and the full history for a changed customer showed the expired and current versions side by side.

**Why Delta Lake.** SCD logic depends on atomic upserts (`MERGE`) and reliable row updates - operations plain CSV/Parquet cannot provide. Delta's transaction log adds ACID guarantees, `MERGE`, and version history, making it a natural fit for slowly changing dimensions and incremental processing.